## Titanic Decision Tree

## **Parte 1**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('./data/train.csv', on_bad_lines='skip')

print("--- Primeiras 5 linhas dos dados: ---")
print(df.head())
print("\n")

# Verificar informações gerais e valores faltantes
print("--- Informações e valores faltantes: ---")
print(df.info())
print("\n")


# Gráfico 1: Proporção de Sobreviventes vs. Não Sobreviventes
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', data=df)
plt.title('Proporção de Sobreviventes (1) e Não Sobreviventes (0)')
plt.show()

# Gráfico 2: Sobrevivência por Sexo
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', hue='Sex', data=df)
plt.title('Sobrevivência por Sexo')
plt.show()

# Gráfico 3: Sobrevivência por Classe do Passageiro
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', hue='Pclass', data=df)
plt.title('Sobrevivência por Classe do Passageiro')
plt.show()

### O que podemos observar da parte 01?

Dados Faltantes: A coluna Age (Idade) tem muitos valores faltando, assim como Cabin. A coluna Embarked (Porto de Embarque) tem poucos valores faltando. É necessário tratar isso.

Padrão de Sobrevivência Geral: Vemos que mais pessoas não sobreviveram (classe 0) do que sobreviveram (classe 1).

Padrão por Sexo: Fica muito claro que mulheres (female) tiveram uma taxa de sobrevivência muito maior que os homens (male). Este é o famoso padrão "mulheres e crianças primeiro".

Padrão por Classe: Passageiros da Primeira Classe (Pclass = 1) tiveram uma chance de sobrevivência muito maior do que os da Segunda e, principalmente, da Terceira Classe.

## **Parte 02**



In [ ]:

# Preencher valores faltantes
# Para 'Age', calculamos a mediana.
df['Age'] = df['Age'].fillna(df['Age'].median())

# Para 'Embarked', calculamos a moda.
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 2. Remover colunas que não ajudam na análise inicial
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True, errors='ignore')

# 3. Codificar colunas de texto para números
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

print("--- Dados após tratamento e codificação: ---")
print(df.head())

### O que podemos observar da parte 02?

Preenchemos as idades faltantes com a idade mediana para não distorcer a distribuição.

Removemos colunas que são identificadores únicos ou que têm informação demais faltando (Cabin).

Codificamos Sex para 0 e 1, uma conversão simples e eficaz.

Codificamos Embarked usando a técnica One-Hot Encoding. Isso cria novas colunas (Embarked_Q, Embarked_S) com valores 0 ou 1, o que é ideal para categorias sem uma relação de ordem entre si.

## **Parte 03**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# 1. Separar os dados em atributos (X) e alvo (y)
X = df.drop('Survived', axis=1)
y = df['Survived']

# 2. Criar e treinar o modelo de Árvore de Decisão
# Usamos max_depth=3 para criar uma árvore simples.
modelo_arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo_arvore.fit(X, y)

# 3. Visualizar a árvore para encontrar as regras
plt.figure(figsize=(20, 12))
tree.plot_tree(modelo_arvore,
               feature_names=X.columns,
               class_names=['Não Sobreviveu', 'Sobreviveu'],
               filled=True,
               rounded=True,
               fontsize=10)
plt.title("Árvore de Decisão - Padrões de Sobrevivência no Titanic")
plt.show()

### O que podemos observar da parte 03?

Regra Principal (Sexo): A primeira e mais importante divisão na árvore é pelo sexo do passageiro. Isso confirma que ser homem ou mulher foi o fator mais determinante para sobreviver.

Padrão para Homens (Mortalidade Alta):

Se o passageiro era um homem (Sex <= 0.5), a chance de mortalidade era altíssima.

Dentro do grupo dos homens, a idade também era um fator: homens com mais de 6.5 anos (Age > 6.5) tiveram uma taxa de sobrevivência muito baixa.

Padrão para Mulheres (Sobrevivência Alta):

Se a passageira era uma mulher (Sex > 0.5), a chance de sobrevivência era muito alta.

Dentro do grupo das mulheres, a classe da passagem (Pclass) era o fator decisivo. Mulheres na 1ª e 2ª classe (Pclass <= 2.5) tiveram uma taxa de sobrevivência altíssima (97%). As da 3ª classe ainda tiveram uma boa chance, mas menor.